### Hopsitexte – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024), 1. Runde, Aufgabe 1*

Zara hat das Texthopsen schon programmiert. Dabei ist ihr aufgefallen, dass die beiden Spieler erstaunlich häufig von derselben Stelle
aus dem Text herausspringen. Sie versucht einen möglichst langen Text zu schreiben, der diese Eigenschaft nicht hat. Das heißt, die
Sprungsequenzen, die am ersten und zweiten Buchstaben des Textes beginnen, sollen nicht zur selben Endposition führen. So einen Text nennt
sie einen ‚Hopsitext‘. Dieser Aufgabentext ist zum Beispiel ein solcher Hopsitext.

#### Aufgabe 1

Schreibe ein Programm, das Zara hilft, möglichst lange und sinnvolle deutsche Hopsitexte zu erstellen. Überlege dir, was das Programm bieten
muss, damit Zara besonders gut damit arbeiten kann.

#### Hintergrund: die Regeln des Texthopsens (aus Junioraufgabe 2)

Jeder Buchstabe hat eine Sprungweite: a = 1, b = 2, …, z = 26, ä = 27, ö = 28, ü = 29, ß = 30. Ein Spieler beginnt beim ersten, der andere
beim zweiten Buchstaben des Textes. Von der Startposition springt man so viele Buchstaben weiter, wie die Sprungweite des Buchstabens
angibt. Alles, was kein Buchstabe ist, wird übersprungen. Das wiederholt man, bis man aus dem Text herausspringt.
Groß- und Kleinbuchstaben haben dieselbe Sprungweite.

Auf dem Aufgabenblatt sind im Aufgabentext oben die Buchstaben farbig markiert, auf denen die beiden Sprungfolgen landen. Sie springen an
verschiedenen Stellen aus dem Text heraus, deshalb ist der Aufgabentext ein Hopsitext.

#### Beispieldaten

Zu dieser Aufgabe gibt es keine eigenen Eingaben. Zur Inspiration kann man die Texte der Junioraufgabe Texthopsen verwenden. Wir haben sie
in den Ordner `beispieldaten` kopiert: `hopsen1.txt` bis `hopsen5.txt`.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation:** Man spielt das Texthopsen mit dem Computer nach. Für jede Spielerin entsteht eine **Liste der Buchstaben**, auf denen sie landet.
- **Kollision:** Landen beide auf **demselben Buchstaben**, sind ihre Sprungfolgen ab dort **gleich**. Sie verlassen den Text dann an derselben Stelle.
  Ein Text ist also genau dann ein Hopsitext, wenn es **keine Kollision** gibt. (Landen sie nie auf demselben Buchstaben, enden sie an verschiedenen
  Stellen.)
- **Nicht-Buchstaben überspringen:** Man bildet eine Liste aller Buchstaben und rechnet mit der **Nummer des Buchstabens**, nicht mit der Stelle im
  Text. Die Stelle im Text merkt man sich nebenbei, um dem Benutzer die Fundstelle zu zeigen.
- **Eine Kollision bleibt:** Springt man in einem Textanfang schon auf denselben Buchstaben, passiert das auch im längeren Text. Wer einen Hopsitext
  Stück für Stück aufbaut, kann also jeden Zwischenstand prüfen und muss ihn nicht verwerfen, solange keine Kollision da ist.
- **Ausprobieren mit Zurückgehen (Backtracking):** Hat man für jeden Abschnitt mehrere Formulierungen, probiert man sie der Reihe nach. Passt
  keine mehr, geht man einen Abschnitt zurück und nimmt dort die nächste Formulierung.

**Nützliche Funktionen und Bibliotheken**
- `str.lower()`, `str.upper()`, Schleifen über Zeichen mit Index, Zeichenketten zerschneiden (`text[von:bis]`)
- `set` (Menge) für die Buchstaben einer Sprungfolge
- `random.Random(zahl)` für wiederholbare Zufallsversuche

## Was soll das Programm bieten?

Zara schreibt den Text selbst, denn er soll **sinnvoll** und deutsch sein. Das Programm soll ihr beim Schreiben **assistieren**, nicht den Text
erzeugen oder Buchstaben löschen. Dafür braucht sie fünf Dinge:

1. **Erkennen:** Ist mein Text ein Hopsitext? Ohne diese Antwort geht nichts.
2. **Verstehen:** Wenn nicht: **Wo** kollidieren die Sprungfolgen, und warum? Zara sieht den Text mit den **markierten Sprüngen**. Wer weiß, welche zwei
   Buchstaben die Kollision verursachen, muss nur dort etwas ändern.
3. **Reparieren:** Welche **Änderungen** an den beiden Stellen beheben die Kollision, ohne eine neue zu erzeugen? Welche **Wörter** (Synonyme, die
   Zara angibt) gehen? So bleibt der Text sinnvoll.
4. **Weiterschreiben:** Welche **Buchstaben oder Wörter** darf ich an den fertigen Text anhängen, ohne dass der Text zerstört wird?
5. **Planen:** Wenn Zara für jeden Satz **mehrere Formulierungen** hat, findet das Programm eine Kombination, die ein Hopsitext ist. So wird der Text
   lang und bleibt sinnvoll.

Wie man das Programm benutzt: Man ruft die Funktionen mit dem Text als Zeichenkette auf (`analysiere(text)` und so weiter). Die Beispiele unten zeigen
jede Funktion an einem Hopsitext und an einem Text mit Kollision.

## Schritt 1: Das Texthopsen simulieren

`sprungweite(zeichen)` gibt die Sprungweite zurück, und 0 für alles, was kein Buchstabe ist. Mit `lower()` sind Groß- und Kleinbuchstaben
gleich. Der Buchstabe `ß` und die Umlaute stehen ausdrücklich in der Tabelle der Aufgabe.

`simuliere(text)` bildet die Liste aller Buchstaben und merkt sich zu jedem Buchstaben seine Sprungweite (`weiten`) und seine **Stelle im Text**
(`plaetze`). Dann berechnet `sprungfolge` die Buchstaben, auf denen eine Spielerin landet, ab einem Startbuchstaben. Bela beginnt beim Buchstaben 0,
Amira beim Buchstaben 1 (beide zählen wir ab 0).

Danach suchen wir die **erste Kollision**: Wir gehen Amiras Sprungfolge durch und nehmen den ersten Buchstaben, auf dem auch Bela landet. Alle
Buchstaben danach sind gemeinsam. Wir merken uns auch die **Buchstaben davor** (die Stellen, von denen die Spielerinnen springen) in `bela_davor` und
`amira_davor`. Landet eine Spielerin schon beim Start auf dem Kollisionsbuchstaben, gibt es davor nichts (`None`).

In [ ]:
def lies_text(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        return f.read()


def sprungweite(zeichen):
    z = zeichen.lower()
    if 'a' <= z <= 'z':
        return ord(z) - ord('a') + 1
    if z == 'ä':
        return 27
    if z == 'ö':
        return 28
    if z == 'ü':
        return 29
    if z == 'ß':
        return 30
    return 0                                               # kein Buchstabe: wird übersprungen


def sprungfolge(weiten, start):
    folge = []
    stelle = start
    while stelle < len(weiten):
        folge.append(stelle)
        stelle = stelle + weiten[stelle]
    return folge


def simuliere(text):
    plaetze = []                                           # Stelle jedes Buchstabens im Text
    weiten = []
    for i in range(len(text)):
        w = sprungweite(text[i])
        if w > 0:
            plaetze.append(i)
            weiten.append(w)
    bela = sprungfolge(weiten, 0)
    amira = sprungfolge(weiten, 1)
    kollision = None
    bela_menge = set(bela)
    for j in range(len(amira)):
        if amira[j] in bela_menge:
            b = amira[j]
            i = bela.index(b)
            kollision = {'buchstabe': b, 'bela_davor': None, 'amira_davor': None}
            if i > 0:
                kollision['bela_davor'] = bela[i - 1]
            if j > 0:
                kollision['amira_davor'] = amira[j - 1]
            break
    return {'plaetze': plaetze, 'weiten': weiten, 'bela': bela, 'amira': amira, 'kollision': kollision}


def ist_hopsitext(text):
    return simuliere(text)['kollision'] is None


beispiel = 'Beispieltexte sind toll'
s = simuliere(beispiel)
print('Bela landet auf den Buchstaben (ab 0):', s['bela'])
print('Amira landet auf den Buchstaben (ab 0):', s['amira'])
print('Kollision:', s['kollision'])
print('Hopsitext?', ist_hopsitext(beispiel))

Bela landet auf den Buchstaben (ab 0): [0, 2, 11]
Amira landet auf den Buchstaben (ab 0): [1, 6, 11]
Kollision: {'buchstabe': 11, 'bela_davor': 2, 'amira_davor': 6}
Hopsitext? False


Das ist der Beispieltext vom Aufgabenblatt der Junioraufgabe: Beide landen auf dem `t` (Buchstabe 11, ab 0 gezählt). Es ist also **kein** Hopsitext, denn die
Sprungfolgen kollidieren. Der Aufgabentext oben dagegen soll einer sein, das prüfen wir später.

## Schritt 2: Verstehen, wo die Kollision liegt

Eine Meldung „kein Hopsitext“ allein hilft Zara wenig. Wir zeigen ihr **die Sprünge im Text**. `zeige(text, von, bis)` druckt den Ausschnitt
von der Stelle `von` bis `bis` des Textes in Zeilen von 60 Zeichen. Unter jeder Zeile steht eine Markierungszeile:

- `B`: Hier landet Bela.
- `A`: Hier landet Amira.
- `K`: Hier landen beide, die **Kollision** und alles danach.

Zeilenumbrüche im Text ersetzen wir zum Anzeigen durch Leerzeichen, damit die Stellen stimmen.

`analysiere(text)` fasst alles zusammen: Anzahl der Buchstaben, das Urteil, bei einer Kollision den Buchstaben, auf dem sie passiert, und die beiden
Buchstaben, von denen die Spielerinnen dorthin gesprungen sind. Stellen im Text zählen wir ab 1, wie ein Mensch.

In [ ]:
def zeige(text, von, bis, breite=60):
    s = simuliere(text)
    marke = {}                                             # Stelle im Text -> B, A oder K
    for b in s['bela']:
        marke[s['plaetze'][b]] = 'B'
    for b in s['amira']:
        stelle = s['plaetze'][b]
        if stelle in marke:
            marke[stelle] = 'K'
        else:
            marke[stelle] = 'A'
    anzeige = text.replace('\n', ' ')
    for anfang in range(von, min(bis, len(text)), breite):
        ende = min(anfang + breite, bis, len(text))
        markierung = ''
        for i in range(anfang, ende):
            if i in marke:
                markierung = markierung + marke[i]
            else:
                markierung = markierung + ' '
        print(anzeige[anfang:ende])
        print(markierung)


def analysiere(text):
    s = simuliere(text)
    n = len(s['plaetze'])
    print('Der Text hat', n, 'Buchstaben und', len(text), 'Zeichen.')
    print('Bela springt', len(s['bela']), 'mal, Amira', len(s['amira']), 'mal.')
    k = s['kollision']
    if k is None:
        print('HOPSITEXT: Die Sprungfolgen treffen sich nie.')
        return
    ort = s['plaetze'][k['buchstabe']]
    print('KEIN Hopsitext: Beide landen auf dem Buchstaben ' + repr(text[ort]) + ' (Zeichen Nr. ' + str(ort + 1) + ' im Text).')
    for name, davor in [('Bela', k['bela_davor']), ('Amira', k['amira_davor'])]:
        if davor is None:
            print('  ' + name + ' startet schon dort.')
        else:
            stelle = s['plaetze'][davor]
            print('  ' + name + ' kommt vom Buchstaben ' + repr(text[stelle]) + ' (Zeichen Nr. ' + str(stelle + 1) + ', Sprungweite ' + str(s['weiten'][davor]) + ').')
    print('Ändere einen dieser Buchstaben, dann ist diese Kollision weg.')
    print('Der Ausschnitt um die Kollision (B: Bela, A: Amira, K: beide):')
    zeige(text, max(0, ort - 40), ort + 20)


hopsen2 = lies_text('beispieldaten/hopsen2.txt')
analysiere(hopsen2)

Der Text hat 221 Buchstaben und 277 Zeichen.
Bela springt 25 mal, Amira 25 mal.
KEIN Hopsitext: Beide landen auf dem Buchstaben 'l' (Zeichen Nr. 52 im Text).
  Bela kommt vom Buchstaben 'd' (Zeichen Nr. 47, Sprungweite 4).
  Amira kommt vom Buchstaben 's' (Zeichen Nr. 28, Sprungweite 19).
Ändere einen dieser Buchstaben, dann ist diese Kollision weg.
Der Ausschnitt um die Kollision (B: Bela, A: Amira, K: beide):
en flog durch das Land; Ein Nilpferd schlummerte im Sand.  D
A     A  B     AA                  B    K             K     


Der Text `hopsen2.txt` (ein Gedicht) ist kein Hopsitext. Der Ausschnitt zeigt es genau: Die Sprungfolgen kollidieren auf dem `l` (Zeichen 52), Bela kommt vom `d` im Wort `Nilpferd`
(Zeichen 47) und Amira vom `s` im Wort `das` (Zeichen 28). Das stimmt mit den offiziellen Lösungshinweisen überein. Zara muss
also nur eines der beiden Zeichen ändern.

## Schritt 3: Beim Reparieren helfen

Wir wissen jetzt, welche zwei Buchstaben das Problem sind. Zara kann sie selbst ändern. Wir helfen ihr dabei auf zwei Arten.

**Ersatzbuchstaben:** `ersatzbuchstaben(text, nr)` probiert für das Zeichen Nummer `nr` jeden anderen Buchstaben aus und gibt die zurück, für die der **ganze** Text
ein Hopsitext wird. (Möglicherweise entsteht an anderer Stelle eine neue Kollision. Deshalb prüfen wir immer den ganzen Text.)

**Wortersatz:** Ein einzelner anderer Buchstabe macht aus einem Wort meist Unsinn. Besser ist ein anderes **Wort**. Zara gibt dem Programm ein Wörterbuch `synonyme`
mit Wörtern und ihren Alternativen. `wortvorschlaege(text, synonyme)` sucht die **Wörter, in denen die beiden Problembuchstaben stehen**, setzt jede Alternative
ein und prüft, ob der Text dann ein Hopsitext ist. So bleibt der Text sinnvoll.

In [ ]:
BUCHSTABEN = 'abcdefghijklmnopqrstuvwxyzäöüß'


def ersatzbuchstaben(text, nr):
    i = nr - 1
    gute = []
    for neu in BUCHSTABEN:
        if text[i].isupper() and neu != 'ß':
            neu = neu.upper()
        if neu != text[i]:
            versuch = text[:i] + neu + text[i + 1:]
            if ist_hopsitext(versuch):
                gute.append(neu)
    return gute


def wort_um(text, i):
    """Anfang und Ende des Wortes, das die Stelle i enthält."""
    anfang = i
    while anfang > 0 and sprungweite(text[anfang - 1]) > 0:
        anfang = anfang - 1
    ende = i + 1
    while ende < len(text) and sprungweite(text[ende]) > 0:
        ende = ende + 1
    return anfang, ende


def wortvorschlaege(text, synonyme):
    s = simuliere(text)
    k = s['kollision']
    if k is None:
        return []
    vorschlaege = []
    for davor in [k['bela_davor'], k['amira_davor']]:
        if davor is not None:
            anfang, ende = wort_um(text, s['plaetze'][davor])
            wort = text[anfang:ende]
            if wort.lower() in synonyme:
                for neu in synonyme[wort.lower()]:
                    if wort[0].isupper():
                        neu = neu[0].upper() + neu[1:]
                    versuch = text[:anfang] + neu + text[ende:]
                    vorschlaege.append((wort, neu, ist_hopsitext(versuch)))
    return vorschlaege


print('Ersatzbuchstaben für Zeichen 47 (d):', ersatzbuchstaben(hopsen2, 47))
print('Ersatzbuchstaben für Zeichen 28 (s):', ersatzbuchstaben(hopsen2, 28))
SYNONYME = {'das': ['dieses', 'jenes', 'ein'], 'nilpferd': ['flusspferd', 'hippo', 'dickhäuter']}
for alt, neu, geht in wortvorschlaege(hopsen2, SYNONYME):
    print(alt, '->', neu, ':', 'Hopsitext!' if geht else 'noch kein Hopsitext')

Ersatzbuchstaben für Zeichen 47 (d): ['a', 'f', 'g', 'i', 'l', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y', 'ä', 'ö']
Ersatzbuchstaben für Zeichen 28 (s): ['d', 'e', 'g', 'h', 'j', 'k', 'n', 'p', 'u', 'v', 'x', 'ä']
Nilpferd -> Flusspferd : noch kein Hopsitext
Nilpferd -> Hippo : Hopsitext!
Nilpferd -> Dickhäuter : Hopsitext!
das -> dieses : Hopsitext!
das -> jenes : noch kein Hopsitext
das -> ein : Hopsitext!


Bei `Nilpferd` lassen sich **Hippo** und **Dickhäuter** einsetzen, bei `das` **dieses** und **ein**, und der ganze Text ist danach ein Hopsitext. **Jenes** und **Flusspferd** dagegen lösen
die Kollision zwar an der alten Stelle, erzeugen aber irgendwo anders eine neue. Das Programm hat das geprüft. Mit einem Satz „durch dieses Land“ bleibt das Gedicht sinnvoll.
Die Ersatzbuchstaben sind dagegen nur ein Hilfsmittel: Ein anderer Buchstabe macht aus einem Wort meist ein anderes, sinnloses.

## Schritt 4: Weiterschreiben

Zara schreibt einen Text Satz für Satz. Dann ist die Frage: **Was darf ich als Nächstes schreiben?** Das Programm hängt dazu jeden Kandidaten an den Text an und prüft, ob der Text
ein Hopsitext bleibt. Das geht schnell, denn wir müssen nur prüfen, nicht raten.

`passende_buchstaben(text)` probiert alle Buchstaben, `passende_woerter(text, kandidaten)` eine Liste von Wörtern oder Satzteilen, die Zara angibt.

In [ ]:
def passende_buchstaben(text):
    gute = []
    for b in BUCHSTABEN:
        if ist_hopsitext(text + b):
            gute.append(b)
    return gute


def passende_woerter(text, kandidaten):
    gute = []
    for wort in kandidaten:
        if ist_hopsitext(text + ' ' + wort):
            gute.append(wort)
    return gute


anfang = 'Heute fuhr unsere Klasse mit dem Bus in den Wald.'
print('Der Anfang ist ein Hopsitext:', ist_hopsitext(anfang))
kandidaten = ['Die Sonne schien und alle waren gut gelaunt.', 'Die Sonne schien, und alle hatten gute Laune.',
              'Es war ein sonniger Tag, und die Stimmung war prächtig.', 'Das Wetter war schön, und alle freuten sich.',
              'Es war warm, und die Sonne schien hell.']
for satz in kandidaten:
    print(ist_hopsitext(anfang + ' ' + satz), ':', satz)
print('passen:', passende_woerter(anfang, kandidaten))

Der Anfang ist ein Hopsitext: True
True : Die Sonne schien und alle waren gut gelaunt.
True : Die Sonne schien, und alle hatten gute Laune.
False : Es war ein sonniger Tag, und die Stimmung war prächtig.
False : Das Wetter war schön, und alle freuten sich.
True : Es war warm, und die Sonne schien hell.
passen: ['Die Sonne schien und alle waren gut gelaunt.', 'Die Sonne schien, und alle hatten gute Laune.', 'Es war warm, und die Sonne schien hell.']


Von den fünf Kandidaten für den zweiten Satz führen nur drei zu einem Hopsitext (der erste, der zweite und der letzte). Das Programm hat Zara also zwei Sackgassen erspart. Wegen der Eigenschaft „Eine Kollision bleibt“ ist es
nie nötig, einen Satz später noch zu ändern, nur weil er einen Satz früher stand.

## Schritt 5: Aus Varianten einen langen Text bauen

Zara schreibt zu jedem Abschnitt der Geschichte **mehrere Formulierungen** auf, alle sinnvoll. Das Programm sucht eine Auswahl, bei der der ganze Text ein Hopsitext ist.

Es geht Abschnitt für Abschnitt vor: Es hängt die erste Formulierung an. Ist der Text ein Hopsitext, kommt der nächste Abschnitt dran. Gibt es für einen Abschnitt keine
passende Formulierung mehr, **geht es zum vorigen Abschnitt zurück** und probiert dort die nächste Formulierung (*Backtracking*). Das ist erlaubt, weil ein Text mit Kollision nie durch
Anhängen besser wird.

`baue_text(abschnitte, stelle, text)` gibt zurück, wie viele Abschnitte der beste gefundene Text umfasst, und den Text. Sobald alle Abschnitte untergebracht sind, hört die Suche auf.

In [ ]:
def baue_text(abschnitte, stelle, text):
    """Ergebnis: (Zahl der verwendeten Abschnitte, Text) für den längsten gefundenen Hopsitext, der mit text beginnt."""
    if stelle == len(abschnitte):
        return stelle, text
    bester = (stelle, text)
    for variante in abschnitte[stelle]:
        neuer_text = (text + ' ' + variante).strip()
        if ist_hopsitext(neuer_text):
            ergebnis = baue_text(abschnitte, stelle + 1, neuer_text)
            if ergebnis[0] > bester[0]:
                bester = ergebnis
            if bester[0] == len(abschnitte):
                return bester                              # alle Abschnitte untergebracht
    return bester


ABSCHNITTE = [
    ['Am Montag fuhr unsere Klasse mit dem Bus in den Wald.', 'Am Montag fuhren wir alle mit dem Bus zum Wald.',
     'Am Dienstag fuhr die ganze Klasse mit dem Zug in den Wald.', 'Heute fuhr unsere Klasse mit dem Bus in den Wald.',
     'Am Freitag fuhren wir mit dem Bus in den Wald.', 'Gestern fuhren wir mit der Klasse in den Wald.'],
    ['Die Sonne schien und alle waren gut gelaunt.', 'Die Sonne schien, und alle hatten gute Laune.',
     'Es war ein sonniger Tag, und die Stimmung war prächtig.', 'Das Wetter war schön, und alle freuten sich.',
     'Es war warm, und die Sonne schien hell.'],
    ['Zuerst wanderten wir zu einem kleinen See.', 'Zuerst gingen wir zu einem kleinen See.',
     'Als Erstes wanderten wir an einen kleinen See.', 'Zuerst wanderten wir an einen stillen See.',
     'Wir gingen zuerst zu einem See im Wald.'],
    ['Dort machten wir eine lange Pause und aßen unsere Brote.', 'Dort rasteten wir und aßen unsere Brote.',
     'Dort legten wir eine Pause ein und packten die Brote aus.', 'Dort ruhten wir uns aus und aßen unsere Brote.',
     'Dort machten wir Rast und aßen Brote.'],
    ['Danach suchte jeder Pilze und Beeren im Moos.', 'Danach suchten wir Pilze und Beeren im Moos.',
     'Anschließend sammelten wir Beeren und Pilze.', 'Danach sammelte jeder Beeren im Wald.',
     'Später suchten wir Pilze im Wald.'],
    ['Am Nachmittag kam ein Gewitter und wir liefen zur Hütte.', 'Am Nachmittag zog ein Gewitter auf, und wir rannten zur Hütte.',
     'Nachmittags kam ein Gewitter, deshalb liefen wir zur Hütte.', 'Dann kam ein Gewitter, und wir rannten schnell zur Hütte.',
     'Plötzlich zog ein Gewitter auf, und wir liefen zur Hütte.'],
    ['Dort tranken wir Tee und sangen alte Lieder.', 'Dort tranken wir heißen Tee und spielten Karten.',
     'In der Hütte tranken wir Tee und erzählten Geschichten.', 'In der Hütte wärmten wir uns bei Tee und Kuchen auf.',
     'Dort gab es Tee, und wir sangen Lieder.'],
    ['Am Abend brachte uns der Bus wieder nach Hause.', 'Am Abend fuhr uns der Bus müde nach Hause.',
     'Abends brachte uns der Bus zurück in die Stadt.', 'Dann fuhren wir müde mit dem Bus nach Hause.',
     'Am Abend waren wir wieder zu Hause.'],
]
anzahl, geschichte = baue_text(ABSCHNITTE, 0, '')
print('Abschnitte untergebracht:', anzahl, 'von', len(ABSCHNITTE))
print(geschichte)
print()
analysiere(geschichte)

Abschnitte untergebracht: 8 von 8
Heute fuhr unsere Klasse mit dem Bus in den Wald. Die Sonne schien und alle waren gut gelaunt. Zuerst wanderten wir zu einem kleinen See. Dort ruhten wir uns aus und aßen unsere Brote. Danach suchte jeder Pilze und Beeren im Moos. Am Nachmittag zog ein Gewitter auf, und wir rannten zur Hütte. Dort tranken wir heißen Tee und spielten Karten. Am Abend brachte uns der Bus wieder nach Hause.

Der Text hat 312 Buchstaben und 390 Zeichen.
Bela springt 25 mal, Amira 26 mal.
HOPSITEXT: Die Sprungfolgen treffen sich nie.


Aus den acht Abschnitten mit je fünf oder sechs Formulierungen hat das Programm einen Hopsitext mit etwa 300 Buchstaben gebaut, den Zara lesen kann. Die Zahl der möglichen Kombinationen ist
groß (6 · 5 · 5 · 5 · 5 · 5 · 5 · 5 = 468 750), aber das Zurückgehen spart fast alle ein: Es prüft immer nur den Anfang, den es schon aufgebaut hat.

## Schritt 6: Die Beispieltexte und der Aufgabentext

Zum Schluss prüfen wir alle Beispieltexte und den Aufgabentext selbst.

In [ ]:
for nr in range(1, 6):
    text = lies_text('beispieldaten/hopsen' + str(nr) + '.txt')
    s = simuliere(text)
    ergebnis = 'Hopsitext'
    if s['kollision'] is not None:
        ergebnis = 'KEIN Hopsitext, Kollision auf Zeichen Nr. ' + str(s['plaetze'][s['kollision']['buchstabe']] + 1)
    print('hopsen' + str(nr) + '.txt:', str(len(s['plaetze'])).rjust(5), 'Buchstaben,', ergebnis)

AUFGABENTEXT = (
    'Zara hat das Texthopsen schon programmiert. Dabei ist ihr aufgefallen, dass die beiden Spieler erstaunlich häufig von '
    'derselben Stelle aus dem Text herausspringen. Sie versucht einen möglichst langen Text zu schreiben, der diese Eigenschaft '
    'nicht hat. Das heißt, die Sprungsequenzen, die am ersten und zweiten Buchstaben des Textes beginnen, sollen nicht zur '
    'selben Endposition führen. So einen Text nennt sie einen ‚Hopsitext‘. Dieser Aufgabentext ist zum Beispiel ein solcher '
    'Hopsitext.')
print()
print('Der Aufgabentext hat', len(simuliere(AUFGABENTEXT)['plaetze']), 'Buchstaben. Hopsitext?', ist_hopsitext(AUFGABENTEXT))
zeige(AUFGABENTEXT, 0, 180)

hopsen1.txt:   836 Buchstaben, Hopsitext
hopsen2.txt:   221 Buchstaben, KEIN Hopsitext, Kollision auf Zeichen Nr. 52
hopsen3.txt:   218 Buchstaben, Hopsitext
hopsen4.txt:   378 Buchstaben, Hopsitext
hopsen5.txt: 10748 Buchstaben, Hopsitext

Der Aufgabentext hat 406 Buchstaben. Hopsitext? True
Zara hat das Texthopsen schon programmiert. Dabei ist ihr au
BAA                     A      B              A A   B       
fgefallen, dass die beiden Spieler erstaunlich häufig von de
A     A          B    A     B   A     A      B         B    
rselben Stelle aus dem Text herausspringen. Sie versucht ein
 A          B           A   B A     B              A      B 


Bis auf `hopsen2.txt` sind alle Beispieltexte Hopsitexte, auch der lange Text `hopsen5.txt` (ein Märchen). Der Aufgabentext von Zara ist ebenfalls einer. Die Markierungszeilen
zeigen die ersten Sprünge von Bela (`B`) und Amira (`A`). Sie landen nie auf demselben Buchstaben.

## Für Neugierige

**Wie selten sind Hopsitexte?** Eine kleine Rechnung zeigt es: Wir würfeln Texte aus Buchstaben mit der Häufigkeit des Märchens `hopsen5.txt` und zählen, wie viele davon Hopsitexte sind.

In [ ]:
import random

buchstaben = []
for zeichen in lies_text('beispieldaten/hopsen5.txt'):
    if sprungweite(zeichen) > 0:
        buchstaben.append(zeichen)
zufall = random.Random(5)
for laenge in [50, 100, 200, 400]:
    hopsitexte = 0
    for versuch in range(500):
        text = ''.join(zufall.choice(buchstaben) for i in range(laenge))
        if ist_hopsitext(text):
            hopsitexte = hopsitexte + 1
    print(laenge, 'Buchstaben:', hopsitexte, 'von 500 zufälligen Texten sind Hopsitexte')

50 Buchstaben: 337 von 500 zufälligen Texten sind Hopsitexte
100 Buchstaben: 238 von 500 zufälligen Texten sind Hopsitexte
200 Buchstaben: 110 von 500 zufälligen Texten sind Hopsitexte
400 Buchstaben: 22 von 500 zufälligen Texten sind Hopsitexte


Je länger der Text, desto seltener ist er ein Hopsitext, denn die beiden Sprungfolgen haben immer wieder die Chance zu kollidieren. Ein langer sinnvoller Hopsitext ist also eine **Auswahlaufgabe**:
Man braucht viele Formulierungen, aus denen das Programm eine passende Kombination findet. Genau das macht unser Programm in Schritt 5.

**Weitere Hilfen, die man bauen könnte:**
- Eine **grafische Oberfläche**, in der Zara den Text tippt und die Markierungen sofort sieht (grün für Bela, blau für Amira, rot für die Kollision).
- Ein **Wörterbuch** mit Synonymen, damit Zara nicht jedes Wort selbst eintragen muss.
- Ein Hinweis, **ab welcher Stelle** der Text fehlerfrei ist, damit Zara dort weiterschreiben kann.
- Eine Suche nach der **kürzesten Änderung**, die einen Text zum Hopsitext macht.

**Warum ist die Kollision der richtige Test?** Springen beide auf denselben Buchstaben, ist die restliche Sprungfolge gleich, und beide verlassen den Text an derselben Stelle. Landen sie dagegen
nie auf demselben Buchstaben, verlassen sie den Text von **verschiedenen** Buchstaben aus. Das ist genau die Forderung der Aufgabe, dass die Sprungfolgen nicht zur selben Endposition führen.

**MiniZinc:** Hier haben wir keinen Abschnitt. Es geht um das Nachspielen und Durchsuchen von Texten, nicht um Bedingungen auf einer festen Zahl von Variablen.